In [1]:
# Setup: DuckDB — zero config, runs in-process, full SQL
# pip install duckdb
import duckdb

# Create an in-memory connection
con = duckdb.connect()

# -------------------------------------------------------
# Seed data: Citi-style server telemetry (employees for ranking demos)
# -------------------------------------------------------

def sql_executer(str):
    con.execute(str)

def sql_executer_printer(str):
    print (con.execute(str).df().to_string(index=False))   

In [2]:
sql = """
CREATE TABLE IF NOT EXISTS employees  AS
SELECT * FROM (VALUES
    ('Alice',   'Engineering', 120000),
    ('Bob',     'Engineering', 120000),
    ('Charlie', 'Engineering', 110000),
    ('Diana',   'Engineering',  95000),
    ('Eve',     'Marketing',   105000),
    ('Frank',   'Marketing',   105000),
    ('Grace',   'Marketing',    88000),
    ('Henry',   'Finance',     115000),
    ('Ivy',     'Finance',     115000),
    ('Jack',    'Finance',      92000)
) t(name, department, salary)
"""
sql_executer(sql)
sql = """
CREATE TABLE IF NOT EXISTS server_metrics AS
SELECT * FROM (VALUES
    ('server-01', DATE '2026-01-01', 55.0),
    ('server-01', DATE '2026-01-02', 62.0),
    ('server-01', DATE '2026-01-03', 58.0),
    ('server-01', DATE '2026-01-04', 71.0),
    ('server-01', DATE '2026-01-05', 89.0),
    ('server-01', DATE '2026-01-06', 94.0),
    ('server-01', DATE '2026-01-07', 88.0),
    ('server-02', DATE '2026-01-01', 40.0),
    ('server-02', DATE '2026-01-02', 45.0),
    ('server-02', DATE '2026-01-03', 43.0),
    ('server-02', DATE '2026-01-04', 50.0),
    ('server-02', DATE '2026-01-05', 48.0),
    ('server-02', DATE '2026-01-06', 52.0),
    ('server-02', DATE '2026-01-07', 55.0)
) t(server_id, collection_date, cpu_utilization)
"""
sql_executer(sql)

In [3]:
sql = """ 
SELECT
    name,
    department,
    salary,
    ROUND(AVG(salary) OVER (PARTITION BY department), 2) AS dept_average
FROM employees;
"""
sql_executer_printer(sql)

   name  department  salary  dept_average
  Henry     Finance  115000     107333.33
    Ivy     Finance  115000     107333.33
   Jack     Finance   92000     107333.33
    Eve   Marketing  105000      99333.33
  Frank   Marketing  105000      99333.33
  Grace   Marketing   88000      99333.33
  Alice Engineering  120000     111250.00
    Bob Engineering  120000     111250.00
Charlie Engineering  110000     111250.00
  Diana Engineering   95000     111250.00


In [4]:
sql=""" 
SELECT 
    name, 
    department, 
    salary,
    DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rank
FROM employees
"""
sql_executer_printer(sql)

   name  department  salary  rank
  Alice Engineering  120000     1
    Bob Engineering  120000     1
Charlie Engineering  110000     2
  Diana Engineering   95000     3
  Henry     Finance  115000     1
    Ivy     Finance  115000     1
   Jack     Finance   92000     2
    Eve   Marketing  105000     1
  Frank   Marketing  105000     1
  Grace   Marketing   88000     2


In [5]:
sql=""" 
SELECT 
    name, 
    department, 
    salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rank
FROM employees
"""
sql_executer_printer(sql)

   name  department  salary  rank
  Henry     Finance  115000     1
    Ivy     Finance  115000     1
   Jack     Finance   92000     3
  Alice Engineering  120000     1
    Bob Engineering  120000     1
Charlie Engineering  110000     3
  Diana Engineering   95000     4
    Eve   Marketing  105000     1
  Frank   Marketing  105000     1
  Grace   Marketing   88000     3


In [6]:
sql=""" 
With ranked_employees as (
SELECT 
    name, 
    department, 
    salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rank
FROM employees ) select name, department, salary, rank from ranked_employees where rank <= 3
"""
sql_executer_printer(sql)

   name  department  salary  rank
  Henry     Finance  115000     1
    Ivy     Finance  115000     1
   Jack     Finance   92000     3
  Alice Engineering  120000     1
    Bob Engineering  120000     1
Charlie Engineering  110000     3
    Eve   Marketing  105000     1
  Frank   Marketing  105000     1
  Grace   Marketing   88000     3


In [7]:
sql=""" 
With ranked_employees as (
SELECT 
    name, 
    department, 
    salary,
    -- 1. The Rank
    DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) as dept_rank,
    
    -- 2. The Average
    ROUND(AVG(salary) OVER (PARTITION BY department), 0) as dept_avg,
    
    -- 3. The Maximum (Who makes the most here?)
    MAX(salary) OVER (PARTITION BY department) as dept_max,
    
    -- 4. The Distance from the Average (Math using a window function!)
    salary - ROUND(AVG(salary) OVER (PARTITION BY department), 2) as diff_from_avg
    
FROM employees ) select name, department, salary,dept_avg, dept_max,diff_from_avg, 
                                        dept_rank from ranked_employees where dept_rank <= 3
"""
sql_executer_printer(sql)

   name  department  salary  dept_avg  dept_max  diff_from_avg  dept_rank
  Alice Engineering  120000  111250.0    120000        8750.00          1
    Bob Engineering  120000  111250.0    120000        8750.00          1
Charlie Engineering  110000  111250.0    120000       -1250.00          2
  Diana Engineering   95000  111250.0    120000      -16250.00          3
  Henry     Finance  115000  107333.0    115000        7666.67          1
    Ivy     Finance  115000  107333.0    115000        7666.67          1
   Jack     Finance   92000  107333.0    115000      -15333.33          2
    Eve   Marketing  105000   99333.0    105000        5666.67          1
  Frank   Marketing  105000   99333.0    105000        5666.67          1
  Grace   Marketing   88000   99333.0    105000      -11333.33          2


In [8]:
sql=""" 
SELECT
    server_id,
    collection_date,
    cpu_utilization as today_cpu,
    
    -- Look backwards 1 row to see yesterday!
    LAG(cpu_utilization, 1) OVER (
        PARTITION BY server_id 
        ORDER BY collection_date
    ) AS yesterday_cpu
    
FROM server_metrics
"""
sql_executer_printer(sql)

server_id collection_date  today_cpu  yesterday_cpu
server-01      2026-01-01       55.0            NaN
server-01      2026-01-02       62.0           55.0
server-01      2026-01-03       58.0           62.0
server-01      2026-01-04       71.0           58.0
server-01      2026-01-05       89.0           71.0
server-01      2026-01-06       94.0           89.0
server-01      2026-01-07       88.0           94.0
server-02      2026-01-01       40.0            NaN
server-02      2026-01-02       45.0           40.0
server-02      2026-01-03       43.0           45.0
server-02      2026-01-04       50.0           43.0
server-02      2026-01-05       48.0           50.0
server-02      2026-01-06       52.0           48.0
server-02      2026-01-07       55.0           52.0


In [9]:
sql=""" 

SELECT
    server_id,
    collection_date,
    cpu_utilization AS today_cpu,
    
    -- This defines the "logic" for the calculation
    LAG(cpu_utilization, 1) OVER (
        PARTITION BY server_id 
        ORDER BY collection_date
    ) AS yesterday_cpu
    
FROM server_metrics

-- This defines the "presentation" for your eyes
ORDER BY server_id, collection_date;

"""
sql_executer_printer(sql)

server_id collection_date  today_cpu  yesterday_cpu
server-01      2026-01-01       55.0            NaN
server-01      2026-01-02       62.0           55.0
server-01      2026-01-03       58.0           62.0
server-01      2026-01-04       71.0           58.0
server-01      2026-01-05       89.0           71.0
server-01      2026-01-06       94.0           89.0
server-01      2026-01-07       88.0           94.0
server-02      2026-01-01       40.0            NaN
server-02      2026-01-02       45.0           40.0
server-02      2026-01-03       43.0           45.0
server-02      2026-01-04       50.0           43.0
server-02      2026-01-05       48.0           50.0
server-02      2026-01-06       52.0           48.0
server-02      2026-01-07       55.0           52.0


In [10]:
sql=""" 
with daily_deltas as (
SELECT
    server_id,
    collection_date,
    cpu_utilization AS today_cpu,
    
    -- This defines the "logic" for the calculation
    cpu_utilization - LAG(cpu_utilization, 1) OVER (
        PARTITION BY server_id 
        ORDER BY collection_date
    ) AS CPU_SPIKE
    
FROM server_metrics

-- This defines the "presentation" for your eyes
ORDER BY server_id, collection_date
) select * from daily_deltas
where CPU_SPIKE > 10

"""
sql_executer_printer(sql)

server_id collection_date  today_cpu  CPU_SPIKE
server-01      2026-01-04       71.0       13.0
server-01      2026-01-05       89.0       18.0


In [11]:
sql=""" 

SELECT 
    server_id,
    collection_date,
    cpu_utilization,
    ROUND(AVG(cpu_utilization) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        -- The Magic Frame:
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ),2) as rolling_7d_avg
    
FROM server_metrics

"""
sql_executer_printer(sql)

server_id collection_date  cpu_utilization  rolling_7d_avg
server-01      2026-01-01             55.0           55.00
server-01      2026-01-02             62.0           58.50
server-01      2026-01-03             58.0           58.33
server-01      2026-01-04             71.0           61.50
server-01      2026-01-05             89.0           67.00
server-01      2026-01-06             94.0           71.50
server-01      2026-01-07             88.0           73.86
server-02      2026-01-01             40.0           40.00
server-02      2026-01-02             45.0           42.50
server-02      2026-01-03             43.0           42.67
server-02      2026-01-04             50.0           44.50
server-02      2026-01-05             48.0           45.20
server-02      2026-01-06             52.0           46.33
server-02      2026-01-07             55.0           47.57


In [12]:
sql="""
CREATE TABLE store_sales AS
SELECT * FROM (VALUES
    ('Store-A', DATE '2026-03-01', 500),
    ('Store-A', DATE '2026-03-02', 200),
    ('Store-A', DATE '2026-03-03', 700),
    ('Store-B', DATE '2026-03-01', 100),
    ('Store-B', DATE '2026-03-02', 300),
    ('Store-B', DATE '2026-03-03', 150)
) t(store_id, sale_date, revenue)
"""
sql_executer(sql)
sql="""
SELECT 
    store_id,
    sale_date,
    revenue as daily_revenue,
    
    -- The Magic!
    SUM(revenue) OVER (
        PARTITION BY store_id
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) as cumulative_revenue
    
FROM store_sales
ORDER BY store_id, sale_date
"""
sql_executer_printer(sql)

store_id  sale_date  daily_revenue  cumulative_revenue
 Store-A 2026-03-01            500               500.0
 Store-A 2026-03-02            200               700.0
 Store-A 2026-03-03            700              1400.0
 Store-B 2026-03-01            100               100.0
 Store-B 2026-03-02            300               400.0
 Store-B 2026-03-03            150               550.0


In [13]:
sql="""
SELECT 
    store_id,
    sale_date,
    revenue as daily_revenue,
    
    -- The Magic!
    SUM(revenue) OVER (
        PARTITION BY store_id
        ORDER BY sale_date
        ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING 
    ) as cumulative_revenue
    
FROM store_sales
ORDER BY store_id, sale_date
"""
sql_executer_printer(sql)

store_id  sale_date  daily_revenue  cumulative_revenue
 Store-A 2026-03-01            500              1400.0
 Store-A 2026-03-02            200               900.0
 Store-A 2026-03-03            700               700.0
 Store-B 2026-03-01            100               550.0
 Store-B 2026-03-02            300               450.0
 Store-B 2026-03-03            150               150.0


In [14]:
sql="""
SELECT 
    store_id,
    sale_date,
    revenue as daily_revenue,
    
    -- The Magic!
    SUM(revenue) OVER (
        PARTITION BY store_id
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) as cumulative_revenue
    
FROM store_sales
ORDER BY store_id, sale_date
"""
sql_executer_printer(sql)

store_id  sale_date  daily_revenue  cumulative_revenue
 Store-A 2026-03-01            500              1400.0
 Store-A 2026-03-02            200              1400.0
 Store-A 2026-03-03            700              1400.0
 Store-B 2026-03-01            100               550.0
 Store-B 2026-03-02            300               550.0
 Store-B 2026-03-03            150               550.0


In [15]:
sql="""
SELECT 
    name, 
    department, 
    salary,
    -- No partition! It just counts every row.
    ROW_NUMBER() OVER (ORDER BY salary DESC) as global_rank
FROM employees
"""
sql_executer_printer(sql)

   name  department  salary  global_rank
  Alice Engineering  120000            1
    Bob Engineering  120000            2
  Henry     Finance  115000            3
    Ivy     Finance  115000            4
Charlie Engineering  110000            5
    Eve   Marketing  105000            6
  Frank   Marketing  105000            7
  Diana Engineering   95000            8
   Jack     Finance   92000            9
  Grace   Marketing   88000           10


In [16]:
sql="""
SELECT 
    name, 
    department, 
    salary,
    -- Resets to 1 for every department!
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as dept_rank
FROM employees

"""
sql_executer_printer(sql)

   name  department  salary  dept_rank
  Alice Engineering  120000          1
    Bob Engineering  120000          2
Charlie Engineering  110000          3
  Diana Engineering   95000          4
  Henry     Finance  115000          1
    Ivy     Finance  115000          2
   Jack     Finance   92000          3
    Eve   Marketing  105000          1
  Frank   Marketing  105000          2
  Grace   Marketing   88000          3


In [17]:
sql="""
SELECT
    name,
    department,
    salary,
    ROW_NUMBER()  OVER (PARTITION BY department ORDER BY salary DESC) AS row_num,
    RANK()        OVER (PARTITION BY department ORDER BY salary DESC) AS rank,
    DENSE_RANK()  OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rank
FROM employees
ORDER BY department, salary DESC
"""
sql_executer_printer(sql)

   name  department  salary  row_num  rank  dense_rank
  Alice Engineering  120000        1     1           1
    Bob Engineering  120000        2     1           1
Charlie Engineering  110000        3     3           2
  Diana Engineering   95000        4     4           3
  Henry     Finance  115000        1     1           1
    Ivy     Finance  115000        2     1           1
   Jack     Finance   92000        3     3           2
    Eve   Marketing  105000        1     1           1
  Frank   Marketing  105000        2     1           1
  Grace   Marketing   88000        3     3           2


In [18]:

# Real interview task: Top 2 earners per department
# DENSE_RANK is correct here — RANK would skip the 3rd person unfairly
sql="""
WITH ranked AS (
    SELECT
        name,
        department,
        salary,
        DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rnk
    FROM employees
)
SELECT name, department, salary, rnk
FROM ranked
WHERE rnk <= 2
ORDER BY department, rnk
"""
print("Top 2 earners per department:")
sql_executer_printer(sql)


Top 2 earners per department:
   name  department  salary  rnk
  Alice Engineering  120000    1
    Bob Engineering  120000    1
Charlie Engineering  110000    2
  Henry     Finance  115000    1
    Ivy     Finance  115000    1
   Jack     Finance   92000    2
    Eve   Marketing  105000    1
  Frank   Marketing  105000    1
  Grace   Marketing   88000    2


In [19]:

# Real Citi use case: detect CPU utilization spikes
# Compare each day's utilization to the previous day — no self-join needed

sql="""
SELECT
    server_id,
    collection_date,
    cpu_utilization,
    LAG(cpu_utilization, 1)  OVER (PARTITION BY server_id ORDER BY collection_date)
        AS prev_day_cpu,
    LEAD(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date)
        AS next_day_cpu,
    ROUND(
        cpu_utilization
        - LAG(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date),
    1) AS day_over_day_delta
FROM server_metrics
ORDER BY server_id, collection_date
"""
sql_executer_printer(sql)

server_id collection_date  cpu_utilization  prev_day_cpu  next_day_cpu  day_over_day_delta
server-01      2026-01-01             55.0           NaN          62.0                 NaN
server-01      2026-01-02             62.0          55.0          58.0                 7.0
server-01      2026-01-03             58.0          62.0          71.0                -4.0
server-01      2026-01-04             71.0          58.0          89.0                13.0
server-01      2026-01-05             89.0          71.0          94.0                18.0
server-01      2026-01-06             94.0          89.0          88.0                 5.0
server-01      2026-01-07             88.0          94.0           NaN                -6.0
server-02      2026-01-01             40.0           NaN          45.0                 NaN
server-02      2026-01-02             45.0          40.0          43.0                 5.0
server-02      2026-01-03             43.0          45.0          50.0                -2.0

In [20]:

# Alert: servers where CPU jumped more than 15% day-over-day
# This is the exact pattern used in Citi APM spike detection
sql="""
WITH daily_deltas AS (
    SELECT
        server_id,
        collection_date,
        cpu_utilization,
        LAG(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date) AS prev_cpu,
        cpu_utilization
        - LAG(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date) AS delta
    FROM server_metrics
)
SELECT server_id, collection_date, prev_cpu, cpu_utilization, ROUND(delta, 1) AS spike
FROM daily_deltas
WHERE delta > 15
ORDER BY spike DESC
"""
sql_executer_printer(sql)


server_id collection_date  prev_cpu  cpu_utilization  spike
server-01      2026-01-05      71.0             89.0   18.0


# Part 3: Rolling Aggregates — Running Totals and Moving Averages

In [21]:

# Rolling 3-day average CPU utilization per server
# ROWS BETWEEN: physical rows (predictable)
# RANGE BETWEEN: logical rows with same ORDER BY value (avoid for time series)

sql = """
SELECT
    server_id,
    collection_date,
    cpu_utilization,
    ROUND(AVG(cpu_utilization) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW   -- 3-day window
    ), 1) AS rolling_3day_avg,
    ROUND(AVG(cpu_utilization) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW   -- 7-day window
    ), 1) AS rolling_7day_avg,
    SUM(cpu_utilization) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW  -- running total
    ) AS running_total_cpu
FROM server_metrics
ORDER BY server_id, collection_date
"""
sql_executer_printer(sql)

server_id collection_date  cpu_utilization  rolling_3day_avg  rolling_7day_avg  running_total_cpu
server-01      2026-01-01             55.0              55.0              55.0               55.0
server-01      2026-01-02             62.0              58.5              58.5              117.0
server-01      2026-01-03             58.0              58.3              58.3              175.0
server-01      2026-01-04             71.0              63.7              61.5              246.0
server-01      2026-01-05             89.0              72.7              67.0              335.0
server-01      2026-01-06             94.0              84.7              71.5              429.0
server-01      2026-01-07             88.0              90.3              73.9              517.0
server-02      2026-01-01             40.0              40.0              40.0               40.0
server-02      2026-01-02             45.0              42.5              42.5               85.0
server-02      2026-

In [22]:
sql = "select * from server_metrics "
sql_executer_printer(sql)

server_id collection_date  cpu_utilization
server-01      2026-01-01             55.0
server-01      2026-01-02             62.0
server-01      2026-01-03             58.0
server-01      2026-01-04             71.0
server-01      2026-01-05             89.0
server-01      2026-01-06             94.0
server-01      2026-01-07             88.0
server-02      2026-01-01             40.0
server-02      2026-01-02             45.0
server-02      2026-01-03             43.0
server-02      2026-01-04             50.0
server-02      2026-01-05             48.0
server-02      2026-01-06             52.0
server-02      2026-01-07             55.0


In [28]:
sql="""
CREATE TABLE IF NOT EXISTS sales AS
SELECT * FROM (VALUES
    (DATE '2026-03-01', 10),
    (DATE '2026-03-02', 20),
    (DATE '2026-03-03', 30),  -- TIE!
    (DATE '2026-03-03', 40),  -- TIE!
    (DATE '2026-03-04', 50)
) t(sale_date, amount)
"""
sql_executer(sql)

sql="""SELECT 
    sale_date, 
    amount,
    SUM(amount) OVER (
        ORDER BY sale_date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) as running_total_ROWS
FROM sales"""
sql_executer_printer(sql)
print('-'*80)

sql="""SELECT 
    sale_date, 
    amount,
    SUM(amount) OVER (
        ORDER BY sale_date 
        RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) as running_total_RANGE
FROM sales

"""
sql_executer_printer(sql)

 sale_date  amount  running_total_ROWS
2026-03-01      10                10.0
2026-03-02      20                30.0
2026-03-03      30                60.0
2026-03-03      40               100.0
2026-03-04      50               150.0
--------------------------------------------------------------------------------
 sale_date  amount  running_total_RANGE
2026-03-01      10                 10.0
2026-03-02      20                 30.0
2026-03-03      30                100.0
2026-03-03      40                100.0
2026-03-04      50                150.0


In [29]:

# EXERCISE 1: Find employees whose salary is above their department average
# Use a window function — no subquery or self-join

result = con.execute("""
WITH dept_avg AS (
    SELECT
        name,
        department,
        salary,
        ROUND(AVG(salary) OVER (PARTITION BY department), 0) AS dept_avg_salary
    FROM employees
)
SELECT name, department, salary, dept_avg_salary,
       salary - dept_avg_salary AS above_avg_by
FROM dept_avg
WHERE salary > dept_avg_salary
ORDER BY department, above_avg_by DESC
""").df()

print("Exercise 1 — above-average earners:")
print(result.to_string(index=False))
# Expected: Alice(120k), Bob(120k) in Eng; Eve+Frank(105k) in Mkt; Henry+Ivy(115k) in Fin

Exercise 1 — above-average earners:
 name  department  salary  dept_avg_salary  above_avg_by
Alice Engineering  120000         111250.0        8750.0
  Bob Engineering  120000         111250.0        8750.0
Henry     Finance  115000         107333.0        7667.0
  Ivy     Finance  115000         107333.0        7667.0
  Eve   Marketing  105000          99333.0        5667.0
Frank   Marketing  105000          99333.0        5667.0


In [30]:

# EXERCISE 2: Find dates where server-01 CPU was trending upward
# (current day higher than previous day AND higher than day before that)

result = con.execute("""
WITH lags AS (
    SELECT
        server_id,
        collection_date,
        cpu_utilization,
        LAG(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date) AS prev_1,
        LAG(cpu_utilization, 2) OVER (PARTITION BY server_id ORDER BY collection_date) AS prev_2
    FROM server_metrics
    WHERE server_id = 'server-01'
)
SELECT collection_date, prev_2, prev_1, cpu_utilization
FROM lags
WHERE cpu_utilization > prev_1 AND prev_1 > prev_2
ORDER BY collection_date
""").df()

print("Exercise 2 — server-01 continuous upward trend:")
print(result.to_string(index=False))
# Expected: rows where CPU is consistently rising over 3 days

Exercise 2 — server-01 continuous upward trend:
collection_date  prev_2  prev_1  cpu_utilization
     2026-01-05    58.0    71.0             89.0
     2026-01-06    71.0    89.0             94.0


In [31]:
# EXERCISE 3: Calculate the percentile rank of each employee's salary
# within their department (0.0 = lowest, 1.0 = highest)
# Hint: use PERCENT_RANK() — same OVER syntax as other window functions

result = con.execute("""
SELECT
    name,
    department,
    salary,
    ROUND(PERCENT_RANK() OVER (
        PARTITION BY department
        ORDER BY salary
    ), 2) AS salary_percentile
FROM employees
ORDER BY department, salary_percentile DESC
""").df()

print("Exercise 3 — salary percentile rank within department:")
print(result.to_string(index=False))
# Expected: top earners have percentile = 1.0

Exercise 3 — salary percentile rank within department:
   name  department  salary  salary_percentile
  Alice Engineering  120000               0.67
    Bob Engineering  120000               0.67
Charlie Engineering  110000               0.33
  Diana Engineering   95000               0.00
  Henry     Finance  115000               0.50
    Ivy     Finance  115000               0.50
   Jack     Finance   92000               0.00
    Eve   Marketing  105000               0.50
  Frank   Marketing  105000               0.50
  Grace   Marketing   88000               0.00


Whaty is PERCENT_RANK()

`PERCENT_RANK()` is an analytical function that tells you exactly "what percentage of the rows are ranked *below* this current row?"

It calculates a value between `0.0` and `1.0`.

-   The lowest value in the partition always gets `0.0` (0% of people are below them).
-   The highest value in the partition always gets `1.0` (100% of people are below them).
-   The median value gets `0.5` (50% of people are below them).

### The Math Behind It

If you have 5 employees grouped in a department, and you order them from lowest salary to highest salary, here is how the math strictly evaluates:

text

Percent_Rank = (Your Rank - 1) / (Total Rows in Partition - 1)

Example Output:

| name | salary | percent_rank (Math) | percent_rank (Decimal) | What it Means |
| --- | --- | --- | --- | --- |
| Dan | $50k | (1-1) / 4 | 0.00 | 0% of people make less than Dan |
| Carl | $60k | (2-1) / 4 | 0.25 | 25% of people make less than Carl |
| Bob | $70k | (3-1) / 4 | 0.50 | 50% of people make less than Bob (Median) |
| Amy | $80k | (4-1) / 4 | 0.75 | 75% of people make less than Amy |
| Zack | $90k | (5-1) / 4 | 1.00 | 100% of people make less than Zack |

### The Real-World Use Case

In Human Resources or Compensation systems, nobody ever says: "Alice is Rank #142 out of 5,000 employees." That number is meaningless.

Instead, they say: "Alice is in the 95th percentile for compensation."

sql

SELECT

  name,

 salary,

  -- Rounding nice and clean

  ROUND(PERCENT_RANK() OVER (ORDER BY salary), 2) as percentile

FROM hr_data

Whenever you see a product feature like: *"You are in the top 10% of Spotify listeners for Taylor Swift"*, the Spotify backend data pipeline used `PERCENT_RANK()` to calculate that!